In [13]:
import os
import requests
import pandas as pd
import numpy as np


sas_token = "sp=rl&st=2026-04-14T11:15:25Z&se=2027-04-14T19:30:25Z&spr=https&sv=2025-11-05&sr=d&sig=EaWZGX93hXkDrMG68%2FcBBsAaPYS49HZn82BRrRozKv4%3D&sdd=2"

list_url = (
    "https://dsaidatalake.dfs.core.windows.net/beehavendata"
    "?directory=bronze/new"
    "&resource=filesystem"
    "&recursive=true&" + sas_token
)

response = requests.get(list_url, timeout=30)
print("Status:", response.status_code)
response.raise_for_status()

if not response.text.strip():
    raise ValueError("Leere Antwort vom Listing-Endpunkt. Bitte SAS/URL pruefen.")

try:
    data = response.json()
except ValueError:
    print("Antwort ist kein JSON. Anfang der Antwort:")
    print(response.text[:500])
    raise

files = [
    item["name"]
    for item in data.get("paths", [])
    if not item.get("isDirectory", False)
]

dfs = {}
for file in files:
    file_url = f"https://dsaidatalake.dfs.core.windows.net/beehavendata/{file}?{sas_token}"
    base_name = os.path.splitext(os.path.basename(file))[0]
    df_name = base_name.split("_")[0] if "_" in base_name else base_name

    print("Lade:", file, "->", df_name)
    dfs[df_name] = pd.read_csv(file_url)



Status: 200
Lade: bronze/new/flow_schwartau.csv -> flow
Lade: bronze/new/humidity_schwartau.csv -> humidity
Lade: bronze/new/temperature_schwartau.csv -> temperature
Lade: bronze/new/weight_schwartau.csv -> weight


In [ ]:

# 1. standardize Colum nnames
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    return df

# 2. convert colums to datetime, that seem to need it :-)
def unify_datetime(df):
    df = df.copy()
    for col in df.columns:
        if "time" in col or "date" in col:
            df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
    return df

# 3. fix data types 
def fix_dtypes(df):
    df = df.copy()
    for col in df.columns:

        # numerisch erzwingen
        if df[col].dtype == "object":
            num = pd.to_numeric(df[col], errors="ignore")
            if num.dtype != "object":
                df[col] = num

        # bool erzwingen
        if df[col].dtype == "object":
            lower = df[col].astype(str).str.lower()
            if set(lower.unique()) <= {"true","false","yes","no","1","0","nan"}:
                df[col] = lower.map({
                    "true": True, "yes": True, "1": True,
                    "false": False, "no": False, "0": False
                })
    return df

# 4. remove duplictes  (NOT for flow!)
def remove_duplicates(df):
    return df.drop_duplicates().reset_index(drop=True)

# 5. remove outliers (z-Score)
def remove_outliers(df, z_thresh=3):
    df = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) == 0:
        return df
    z = (df[numeric_cols] - df[numeric_cols].mean()) / df[numeric_cols].std()
    mask = (z.abs() <= z_thresh).all(axis=1)
    return df[mask].reset_index(drop=True)

# 6. flow: determine departure/arrival 
def process_flow(df):
    df = df.copy()
    df = df.sort_values("timestamp")
    df["occurrence"] = df.groupby("timestamp").cumcount()
    df["flow_type"] = df["occurrence"].map({0: "departure", 1: "arrival"})
    df = df.drop(columns=["occurrence"])
    return df

# 7) pivot flow
def pivot_flow(df):
    df = df.copy()
    silver = df.pivot_table(
        index="timestamp",
        columns="flow_type",
        values="flow",
        aggfunc="first"
    ).reset_index()
    silver["netto_flow"] = silver["departure"] - silver["arrival"]
    return silver

# 8. LOOP over all DataFrames with fixes calculatoins for silver
dfs_silver = {}

for name, df in dfs.items():

    # generische Cleanings
    df2 = clean_columns(df)
    df2 = unify_datetime(df2)
    df2 = fix_dtypes(df2)

    # no duplicates for flow , but pivot depatartuer / arrival
    if name == "flow":
        df2 = process_flow(df2)
        df2 = pivot_flow(df2)

        # NOW you can chect for outlieres
        df2 = remove_outliers(df2)

    else:
        # humidity, temperature, weight → normal 
        df2 = remove_duplicates(df2)
        df2 = remove_outliers(df2)

    dfs_silver[name] = df2




In [19]:
# Ergebnis:
dfs_silver["flow"].loc[dfs_silver["flow"]["netto_flow"] != 0]
#dfs_silver['humidity']
#dfs_silver['temperature']
#dfs_silver['weight']

flow_type,timestamp,arrival,departure,netto_flow
271,2017-01-01 18:46:00+00:00,0,-1,-1
272,2017-01-01 18:47:00+00:00,1,0,-1
276,2017-01-01 18:51:00+00:00,0,-1,-1
322,2017-01-01 19:37:00+00:00,0,-1,-1
364,2017-01-01 20:19:00+00:00,0,-1,-1
...,...,...,...,...
1202431,2019-05-31 13:54:00+00:00,149,-135,-284
1202432,2019-05-31 13:55:00+00:00,-135,127,262
1202433,2019-05-31 13:56:00+00:00,-144,146,290
1202434,2019-05-31 13:57:00+00:00,-144,151,295
